In [1]:
import tskit
import msprime
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from utils import *

/home/bjarkemp/miniforge3/envs/recapitation/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Simulate large ARG (n=500)

In [2]:
seed = 122439.29573941
print(seed)
ts = msprime.sim_ancestry(
    500, recombination_rate=1e-6, sequence_length=1e6,
    random_seed=int(seed), record_full_arg=True, model='smc')

122439.29573941


## Recombination summary

In [3]:
df = recombination_summary(ts)
df

,rec_number,child,descendant_samples,breakpoint,time,tree_index_left,tree_index_right,coal_left,coal_right
0,1,65,[65],521373.0,0.000318,23,24,12.0,14.0
1,2,102,[102],61211.0,0.000819,5,6,20.0,25.0
2,3,820,[820],582220.0,0.001606,26,27,20.0,9.0
3,4,151,[151],99883.0,0.001729,6,7,19.0,19.0
4,5,43,[43],472828.0,0.002077,21,22,15.0,14.0
5,6,1627,"[255, 732, 860]",268260.0,0.007641,14,15,15.0,4.0
6,7,1547,"[69, 229, 565, 661, 808]",928984.0,0.007846,36,37,13.0,12.0
7,8,914,[914],580637.0,0.009737,25,26,4.0,5.0
8,9,1620,"[278, 905]",755738.0,0.011376,30,31,16.0,15.0
9,10,1821,"[414, 419, 453, 472, 578, 713, 748, 824, 829, ...",751098.0,0.017313,28,29,17.0,17.0


## Run the inference

In [4]:
mts = add_mutations(ts)

In [5]:
incomp_matrix_fast = compute_incompatibility_matrix_batched(mts, recsites=find_recsites(mts))

Computing Matrix: 100%|██████████| 40/40 [02:11<00:00,  3.30s/it]


In [6]:
incomp_matrix = compute_incompatibility_matrix(mts, recsites=find_recsites(mts))

Computing incompatibility: 100%|██████████| 39960/39960 [3:34:53<00:00,  3.10it/s]  


In [ ]:
coal_nodes = np.where(mts.tables.nodes.flags == 0)[0]
nodes = np.concatenate([mts.samples(), coal_nodes])
genotypes = mts.genotype_matrix(samples=nodes, isolated_as_missing=False)
incompatible_sites = np.where(incomp_matrix.any(axis=1))[0]
incomp_genotypes = genotypes[incompatible_sites, :].T

: 

In [ ]:
sub_incomp = incomp_matrix[np.ix_(incompatible_sites, incompatible_sites)]
all_genos, pairs = incomp_pair_genotypes(incomp_genotypes, sub_incomp)

In [ ]:
df = get_incomp_site_position(mts, incomp_matrix, df)
df

## Compare scoring functions

In [ ]:
n_sites = incomp_genotypes.shape[1]
n_ind = incomp_genotypes.shape[0]

cross_genos = [cross_point_genotypes(all_genos, pairs, point) for point in range(1, n_sites)]

# Compute score matrices
score_orig = np.zeros((n_sites - 1, n_ind))
score_invfreq = np.zeros((n_sites - 1, n_ind))
score_logfreq = np.zeros((n_sites - 1, n_ind))
score_rarest_w0 = np.zeros((n_sites - 1, n_ind))
score_rarest_w05 = np.zeros((n_sites - 1, n_ind))
score_rarest_w1 = np.zeros((n_sites - 1, n_ind))

for i, cg in enumerate(tqdm(cross_genos, desc='Scoring')):
    if cg.shape[0] > 0:
        score_orig[i] = score_individuals(cg, w=0)
        score_invfreq[i] = score_inverse_frequency(cg)
        score_logfreq[i] = score_log_frequency(cg)
        score_rarest_w0[i] = score_rarest_gamete(cg, w=0)
        score_rarest_w05[i] = score_rarest_gamete(cg, w=0.5)
        score_rarest_w1[i] = score_rarest_gamete(cg, w=1)

# Plot all
fig, axes = plt.subplots(6, 1, figsize=(14, 26), sharex=True)

all_scores = [score_orig, score_invfreq, score_logfreq,
              score_rarest_w0, score_rarest_w05, score_rarest_w1]
all_titles = ['Original (w=0, +1 for 11)',
              'Inverse frequency (1/count)',
              'Log frequency (-log(count/n))',
              'Rarest-gamete (w=0, +1 for 11 only)',
              'Rarest-gamete (w=0.5, +1 for 11, +0.5 for rarest)',
              'Rarest-gamete (w=1, +1 for 11, +1 for rarest)']

for ax, scores, title in zip(axes, all_scores, all_titles):
    for ind in range(n_ind):
        ax.scatter(range(1, n_sites), scores[:, ind], alpha=0.1, s=5, c='steelblue')

    for _, row in df.iterrows():
        ax.axvline(row['incomp_split_point'], color='red', linestyle='--', alpha=0.5)

    ax.set_ylabel('Score')
    ax.set_title(title)

axes[-1].set_xlabel('Split point (between incompatible site i-1 and i)')
plt.tight_layout()
plt.show()